# Binary Vessel Segmentation on FIVES Dataset using Implicit Neural Representations

This notebook implements a **SIREN-based Implicit Neural Representation (INR)** model for
binary retinal vessel segmentation on the [FIVES dataset](https://figshare.com/articles/figure/FIVES_A_Fundus_Image_Dataset_for_AI-based_Vessel_Segmentation/19688169).

## Key Idea
Instead of using convolutional neural networks that operate on image patches directly,
INR models learn a **continuous function** that maps pixel coordinates and intensity
to segmentation labels:

$$f: (x, y, I) \rightarrow \{\text{background}, \text{vessel}\}$$

This allows **resolution-independent** inference — the model can segment images at
any resolution by simply querying with a denser coordinate grid.

## Architecture
1. **Positional Encoding**: Fourier features to overcome spectral bias
2. **SIREN MLP**: Sine-activated hidden layers with batch normalization
3. **Focal-Dice Loss**: Combined loss for handling class imbalance

## Dataset
- **FIVES**: 800 high-resolution (2048×2048) color fundus photographs
- Binary vessel masks for supervised training
- Images are split into 256×256 patches for memory-efficient training

## 1. Imports and Configuration

In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

# ── Configuration ─────────────────────────────────────────────────────────────
PATCH_SIZE = 256              # Side length of training patches (pixels)
FULL_IMAGE_SIZE = 2048        # FIVES images are 2048×2048
BATCH_SIZE = 128              # Number of patches per training batch
NUM_EPOCHS = 50               # Training epochs
LEARNING_RATE = 1e-1          # Initial learning rate for Adam
NUM_CLASSES = 2               # Binary: vessel vs. background
NUM_LAYERS = 4                # Depth of the SIREN MLP
HIDDEN_DIM = PATCH_SIZE       # Hidden layer width (matched to patch size)
NUM_FREQS = 5                 # Positional encoding frequency bands
ALPHA = 0.75                  # Focal loss alpha (positive class weight)
GAMMA = 2                     # Focal loss gamma (focusing parameter)
SMOOTH = 1e-5                 # Dice loss smoothing constant
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## 2. Dataset

The `SegmentationDataset` class handles:
- Loading 2048×2048 fundus images and splitting them into 256×256 patches
- Data augmentation (flips, rotations, CLAHE, noise)
- Converting each patch into **(x, y, intensity)** feature vectors for the INR model

Each pixel becomes a training sample: its normalized coordinates (x, y ∈ [0,1]) and
grayscale intensity are the input features, and the binary vessel label is the target.

In [ ]:
class SegmentationDataset(Dataset):
    """FIVES fundus image dataset for binary vessel segmentation with INR.

    Splits high-resolution 2048×2048 fundus images into fixed-size patches
    and generates per-pixel (coordinate, intensity) -> label pairs.

    Args:
        images_path: Path to directory of fundus images.
        masks_path: Path to directory of binary vessel masks.
        patch_size: Side length of square patches.
        augment: Whether to apply data augmentation.
    """

    VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

    def __init__(self, images_path, masks_path, patch_size=256, augment=True):
        self.images_path = images_path
        self.masks_path = masks_path
        self.patch_size = patch_size
        self.augment = augment

        # Filter out hidden files (e.g., .DS_Store on macOS)
        self.image_files = sorted([
            f for f in os.listdir(images_path)
            if not f.startswith('.') and f.lower().endswith(self.VALID_EXTENSIONS)
        ])
        self.mask_files = sorted([
            f for f in os.listdir(masks_path)
            if not f.startswith('.') and f.lower().endswith(self.VALID_EXTENSIONS)
        ])

        # Augmentation pipeline — conservative transforms to preserve vessel topology
        self.augmentations = A.Compose(
            [
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=15, p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.GaussianBlur(blur_limit=3, p=0.2),
                A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
                A.CLAHE(clip_limit=2, p=0.3),  # Contrast-limited adaptive histogram equalization
            ],
            additional_targets={'mask': 'image'},
        )

    def __len__(self):
        """Total number of patches across all images."""
        patches_per_image = (FULL_IMAGE_SIZE // self.patch_size) ** 2
        return len(self.image_files) * patches_per_image

    def __getitem__(self, idx):
        """Return one patch as (coords_intensities, labels).

        The flat index encodes both which image and which patch:
            image_idx = idx // patches_per_image
            patch_idx = idx % patches_per_image
        """
        patches_per_image = (FULL_IMAGE_SIZE // self.patch_size) ** 2
        image_idx = idx // patches_per_image
        patch_idx = idx % patches_per_image

        # Load full grayscale image and binary mask
        image_path = os.path.join(self.images_path, self.image_files[image_idx])
        mask_path = os.path.join(self.masks_path, self.mask_files[image_idx])
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.uint8) / 255

        # Compute patch position in the image grid
        grid_cols = FULL_IMAGE_SIZE // self.patch_size
        row = patch_idx // grid_cols
        col = patch_idx % grid_cols
        y_start, x_start = row * self.patch_size, col * self.patch_size

        # Extract the patch
        image_patch = image[y_start:y_start + self.patch_size, x_start:x_start + self.patch_size]
        mask_patch = mask[y_start:y_start + self.patch_size, x_start:x_start + self.patch_size]

        # Apply augmentation jointly to image and mask
        if self.augment:
            augmented = self.augmentations(image=image_patch, mask=mask_patch)
            image_patch, mask_patch = augmented['image'], augmented['mask']

        # Convert patch pixels to (x, y, intensity) vectors for the INR model
        coords_intensities = self.generate_coords_intensities(image_patch)
        labels = torch.tensor(mask_patch.flatten(), dtype=torch.long)

        return coords_intensities, labels

    def generate_coords_intensities(self, image_patch, device=None):
        """Convert a 2D image patch to (x_norm, y_norm, intensity) feature vectors.

        Creates a meshgrid of pixel positions normalized to [0, 1], then appends
        each pixel's intensity as a third feature dimension.

        Args:
            image_patch: 2D array of shape (H, W) with values in [0, 1].
            device: Optional torch device to move the result to.

        Returns:
            Tensor of shape (H*W, 3) — one row per pixel.
        """
        if isinstance(image_patch, torch.Tensor):
            image_patch = image_patch.cpu().numpy()

        # Handle multi-channel inputs by converting to single channel
        if image_patch.ndim == 3:
            if image_patch.shape[0] in (1, 3):
                image_patch = np.mean(image_patch, axis=0)

        H, W = image_patch.shape

        # Create a normalized coordinate grid: (x, y) ∈ [0, 1]
        y_coords, x_coords = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        coords = np.stack((x_coords, y_coords), axis=-1).reshape(-1, 2).astype(np.float32)
        coords /= np.array([W - 1, H - 1], dtype=np.float32)

        # Append pixel intensity as the third feature
        intensities = image_patch.flatten().reshape(-1, 1)
        coords_intensities = np.concatenate([coords, intensities], axis=-1)

        tensor = torch.tensor(coords_intensities, dtype=torch.float32)
        if device is not None:
            tensor = tensor.to(device)

        return tensor

### 2.1 Initialize Dataset and DataLoader

In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
# Update these paths to point to your local FIVES dataset
IMAGES_PATH = 'data/FIVES/train/Original'
MASKS_PATH = 'data/FIVES/train/Ground truth'

# Create dataset (augmentation disabled for reproducibility during development)
dataset = SegmentationDataset(IMAGES_PATH, MASKS_PATH, patch_size=PATCH_SIZE, augment=False)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Verify shapes: coords_intensities should be (batch, H*W, 3), labels (batch, H*W)
sample_coords, sample_labels = next(iter(dataloader))
print(f"Coords+Intensity shape: {sample_coords.shape}")
print(f"Labels shape: {sample_labels.shape}")

## 3. Model Architecture

### Positional Encoding
Raw (x, y) coordinates are low-dimensional and suffer from *spectral bias* —
MLPs struggle to learn high-frequency spatial patterns. Fourier feature positional
encoding maps coordinates to a higher-dimensional space using sinusoids:

$$\gamma(x) = [\sin(2^0 x), \cos(2^0 x), \sin(2^1 x), \cos(2^1 x), \dots, \sin(2^{L-1} x), \cos(2^{L-1} x)]$$

### SIREN (Sinusoidal Representation Network)
Each hidden layer applies: $h_{i+1} = \sin(\omega_0 \cdot W_i h_i + b_i)$

The sine activation is better suited for representing fine spatial details
(thin vessels, capillaries) compared to ReLU. Weight initialization follows
Sitzmann et al. (2020) to maintain activation variance through depth.

In [ ]:
class PositionalEncoding(nn.Module):
    """Fourier feature encoding for 2D spatial coordinates.

    Maps (x, y) to [sin(2^0*x), cos(2^0*x), ..., sin(2^(L-1)*x), cos(2^(L-1)*x)]
    for each coordinate dimension.

    Output dimension = num_coords × num_freqs × 2
    """

    def __init__(self, num_freqs):
        super().__init__()
        self.num_freqs = num_freqs

    def forward(self, x):
        # Exponentially spaced frequencies: [2^0, 2^1, ..., 2^(L-1)]
        frequencies = torch.linspace(0, self.num_freqs - 1, self.num_freqs, device=x.device)
        frequencies = 2.0 ** frequencies                   # Shape: (num_freqs,)
        frequencies = frequencies.view(1, 1, -1)           # Shape: (1, 1, num_freqs)

        x = x.unsqueeze(-1)                                # (N, num_coords, 1)
        x = x * frequencies                                # (N, num_coords, num_freqs) via broadcasting
        x = torch.cat([torch.sin(x), torch.cos(x)], dim=-1)  # (N, num_coords, 2*num_freqs)
        x = x.view(x.shape[0], -1)                        # Flatten: (N, num_coords * 2 * num_freqs)
        return x


class AdaptiveDropout(nn.Module):
    """Dropout with exponentially decaying rate.

    Starts aggressive (high p) to regularize early training,
    then relaxes as the model converges.
    """

    def __init__(self, initial_p=0.5, decay_factor=0.95):
        super().__init__()
        self.p = initial_p
        self.decay_factor = decay_factor

    def forward(self, x):
        if self.training:
            return F.dropout(x, p=self.p, training=True)
        return x

    def step(self):
        """Call at the end of each epoch to decay dropout probability."""
        self.p *= self.decay_factor


class SineLayer(nn.Module):
    """One SIREN layer: Linear transform + sinusoidal activation.

    Uses the initialization scheme from Sitzmann et al. (NeurIPS 2020)
    to ensure stable gradient flow through deep sine networks.

    Args:
        in_features: Input dimension.
        out_features: Output dimension.
        is_first: Use first-layer initialization (uniform in [-1/n, 1/n]).
        omega_0: Frequency scaling for sine activation.
    """

    def __init__(self, in_features, out_features, bias=True, is_first=False, omega_0=1.0):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.in_features = in_features
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self._init_weights()

    def _init_weights(self):
        """SIREN-specific weight initialization.

        First layer:  W ~ U(-1/n, 1/n)
        Other layers: W ~ U(-sqrt(6/n)/ω₀, sqrt(6/n)/ω₀)

        This ensures unit variance of activations at each layer.
        """
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.in_features
            else:
                bound = np.sqrt(6.0 / self.in_features) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class INRSegmentationModel(nn.Module):
    """Implicit Neural Representation model for binary vessel segmentation.

    Pipeline:
        (x, y, intensity) → Positional Encoding → Reduction → SIREN MLP → σ(output)

    Key property: resolution-independent — trained on 256×256 patches,
    can infer at any resolution by varying the coordinate grid density.

    Args:
        num_classes: Number of output classes (2 for binary segmentation).
        hidden_dim: Width of hidden layers.
        num_layers: Total number of MLP layers.
        num_freqs: Positional encoding frequency bands.
        outermost_linear: Use linear (not sine) output layer.
    """

    def __init__(self, num_classes=2, hidden_dim=256, num_layers=5, num_freqs=10,
                 initial_dropout_p=0.5, outermost_linear=False, linear_network=False):
        super().__init__()
        self.num_classes = num_classes
        self.outermost_linear = outermost_linear

        # Positional encoding: (x, y) → high-dimensional Fourier features
        self.pos_enc = PositionalEncoding(num_freqs)
        num_coords = 2  # (x, y)
        input_dim = num_coords * num_freqs * 2 + 1  # +1 for pixel intensity

        # Project encoded features to hidden dimension
        self.reduction_layer = nn.Linear(input_dim, hidden_dim)

        # Adaptive dropout for regularization
        self.dropouts = nn.ModuleList(
            [AdaptiveDropout(initial_dropout_p) for _ in range(num_layers - 1)]
        )

        # Build the MLP backbone
        self.mlp = nn.ModuleList()

        # First layer (special initialization)
        self.mlp.append(nn.Sequential(
            SineLayer(hidden_dim, hidden_dim, is_first=True),
            nn.BatchNorm1d(hidden_dim),  # Stabilize training
        ))

        # Intermediate layers
        for _ in range(1, num_layers - 2):
            if linear_network:
                # Ablation: ReLU layers instead of sine
                self.mlp.append(nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                ))
            else:
                self.mlp.append(nn.Sequential(
                    SineLayer(hidden_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                ))

        # Output layer: single unit with sigmoid for binary probability
        if outermost_linear:
            self.mlp.append(nn.Linear(hidden_dim, 1))
        else:
            self.mlp.append(SineLayer(hidden_dim, 1))

    def forward(self, coords_intensities):
        """Map per-pixel (x, y, intensity) to vessel probability.

        Args:
            coords_intensities: (N, 3) tensor — [x_norm, y_norm, intensity].

        Returns:
            (N, 1) tensor of vessel probabilities in [0, 1].
        """
        # Split coordinates and intensity
        coords = coords_intensities[:, :-1]          # (N, 2)
        intensities = coords_intensities[:, -1:]      # (N, 1)

        # Encode coordinates with Fourier features
        x = self.pos_enc(coords)                      # (N, num_coords * 2 * num_freqs)

        # Concatenate with raw intensity
        x = torch.cat([x, intensities], dim=-1)       # (N, input_dim)

        # Project to hidden dimension
        x = self.reduction_layer(x)                   # (N, hidden_dim)

        # Forward through SIREN layers with dropout
        for i, layer in enumerate(self.mlp[:-1]):
            x = layer(x)
            x = self.dropouts[i](x)

        # Output layer + sigmoid activation
        x = self.mlp[-1](x)
        x = torch.sigmoid(x)
        return x


print(f"Using device: {DEVICE}")

## 4. Loss Function

We use a **Focal-Dice combined loss** to address the severe class imbalance
in retinal images (vessels occupy only ~5-15% of pixels).

- **Focal Loss**: Down-weights well-classified pixels, focusing on hard examples
  (thin vessels, boundaries). Controlled by α (class weight) and γ (focusing strength).
- **Dice Loss**: Directly optimizes the F1/Dice coefficient, which measures
  spatial overlap between prediction and ground truth.

In [ ]:
class FocalDiceLoss(nn.Module):
    """Combined Focal + Dice loss for binary segmentation.

    Total Loss = Focal_Loss + Dice_Loss

    Args:
        alpha: Weight for positive class in focal loss (higher = more focus on vessels).
        gamma: Focusing parameter (higher = more emphasis on hard examples).
        smooth: Smoothing constant to prevent division by zero in Dice.
    """

    def __init__(self, alpha=0.8, gamma=2, smooth=1e-5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Clamp predictions for numerical stability
        inputs = torch.clamp(inputs, 1e-7, 1 - 1e-7)

        # ── Focal Loss ────────────────────────────────────────────────────────
        # Positive pixels: -α(1-p)^γ log(p)  |  Negative: -(1-α)p^γ log(1-p)
        focal_loss = (
            -self.alpha * (1 - inputs) ** self.gamma * targets * torch.log(inputs)
            - (1 - self.alpha) * inputs ** self.gamma * (1 - targets) * torch.log(1 - inputs)
        )
        focal_loss = focal_loss.mean()

        # ── Dice Loss ─────────────────────────────────────────────────────────
        # Dice = 2|A∩B| / (|A| + |B|)  →  Loss = 1 - Dice
        intersection = (inputs * targets).sum()
        union = inputs.sum() + targets.sum()
        dice_loss = 1 - (2.0 * intersection + self.smooth) / (union + self.smooth)

        return focal_loss + dice_loss

## 5. Training Loop

Training strategy:
- **Optimizer**: Adam with aggressive initial LR (0.1)
- **Scheduler**: ReduceLROnPlateau — reduces LR by 0.3× when loss plateaus for 5 epochs
- **Checkpointing**: Best model saved based on epoch loss

In [ ]:
# ── Model instantiation ───────────────────────────────────────────────────────
model = INRSegmentationModel(
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_freqs=NUM_FREQS,
    outermost_linear=False,
    linear_network=False,
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = FocalDiceLoss(alpha=ALPHA, gamma=GAMMA, smooth=SMOOTH).to(DEVICE)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=5, verbose=True)

# ── Training loop ─────────────────────────────────────────────────────────────
losses = []
best_loss = float('inf')
MODEL_SAVE_PATH = 'checkpoints/fives_binary_4layers.pth'
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    for coords_intensities, labels in dataloader:
        coords_intensities = coords_intensities.to(DEVICE)
        labels = labels.to(DEVICE)

        # Flatten batch: (batch_size, H*W, 3) → (batch_size*H*W, 3)
        coords_intensities = coords_intensities.view(-1, 3)
        labels = labels.view(-1)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(coords_intensities).squeeze(-1)  # (N, 1) → (N,)

        # Compute combined Focal-Dice loss
        loss = criterion(outputs, labels)

        # Backward pass and weight update
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Average loss for this epoch
    epoch_loss /= len(dataloader)
    losses.append(epoch_loss)

    # Step the learning rate scheduler
    scheduler.step(epoch_loss)

    # Log progress
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}, LR: {current_lr:.6f}')

    # Save best model checkpoint
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'  → New best model saved (loss: {best_loss:.4f})')

### 5.1 Training Loss Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Focal-Dice Loss')
plt.title('Training Loss over Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluation

For inference, we:
1. Split the test image into patches (same size as training)
2. Generate coordinate-intensity pairs for each patch
3. Run the model on each patch independently
4. Reconstruct the full segmentation mask from patch predictions
5. Apply a 0.5 threshold for binary classification

In [ ]:
def split_into_patches(image, patch_size):
    """Split a single-channel image tensor into non-overlapping patches."""
    _, H, W = image.shape
    patches = []
    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            patches.append(image[:, i:i + patch_size, j:j + patch_size])
    return patches


def reconstruct_from_patches(patches, image_shape, patch_size):
    """Reassemble patches into a full image, averaging overlapping regions."""
    H, W = image_shape
    reconstructed = torch.zeros((H, W), device=patches[0].device)
    count_map = torch.zeros((H, W), device=patches[0].device)

    idx = 0
    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            h, w = patches[idx].shape[1:]
            reconstructed[i:i + h, j:j + w] += patches[idx][0].to(reconstructed.device)
            count_map[i:i + h, j:j + w] += 1
            idx += 1

    count_map = torch.clamp(count_map, min=1)
    return reconstructed / count_map


def evaluate_model(model, dataset, image, mask, patch_size, criterion):
    """Run inference on a test image and display results side-by-side."""
    model.eval()

    image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    image_tensor = image_tensor.squeeze(0)  # (1, H, W)

    patches = split_into_patches(image_tensor, patch_size)
    predicted_patches = []
    losses = []

    for img_patch in patches:
        H, W = img_patch.shape[1:]
        coords_intensities = dataset.generate_coords_intensities(img_patch, device=DEVICE)

        with torch.no_grad():
            outputs = model(coords_intensities).squeeze(-1).view(H, W)
            predicted_labels = (outputs > 0.5).long()
            predicted_patches.append(predicted_labels.unsqueeze(0))

            loss = criterion(outputs.view(-1), img_patch.reshape(-1))
            losses.append(loss.item())

    # Reconstruct full prediction and threshold
    predicted_mask = reconstruct_from_patches(
        predicted_patches, image_tensor.shape[1:], patch_size
    ).cpu().numpy()
    predicted_mask = (predicted_mask > 0.5).astype(np.uint8)
    total_loss = sum(losses)

    # Side-by-side visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Ground Truth Mask')
    axes[1].axis('off')

    axes[2].imshow(predicted_mask, cmap='gray')
    axes[2].set_title(f'Predicted Mask (Loss: {total_loss:.4f})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    return predicted_mask, total_loss

In [ ]:
# ── Run evaluation on a test image ────────────────────────────────────────────
TEST_IMAGE_PATH = 'data/FIVES/test/Original/1_A.png'
TEST_MASK_PATH = 'data/FIVES/test/Ground truth/1_A.png'

test_image = cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_LINEAR)

test_mask = cv2.imread(TEST_MASK_PATH, cv2.IMREAD_GRAYSCALE)
test_mask = cv2.resize(test_mask, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)

predicted_mask, loss = evaluate_model(
    model, dataset, test_image, test_mask, patch_size=64, criterion=criterion
)

## 7. Results Summary

| Configuration | Layers | α | γ | Loss (50 epochs) |
|---|---|---|---|---|
| SIREN + FocalDice | 4 | 0.75 | 2 | **0.8846** |
| SIREN + FocalDice | 6 | 0.75 | 2 | 0.9217 (9 epochs) |
| SIREN + FocalDice | 2 | 0.75 | 2 | 0.8762 (9 epochs) |

**Key observations:**
- 4-layer SIREN achieves the best convergence behavior
- Deeper networks (6 layers) overfit more quickly
- The Focal-Dice loss effectively handles the ~85/15% class imbalance